In [0]:
%fs
ls dbfs:/

In [0]:
%fs
ls /Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA.csv

In [0]:
%fs
ls /Volumes/workspace/default/arquivos-aula/Anac/

In [0]:
caminho_csv = "dbfs:/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA.csv"

df_csv = (
    spark.read
    .format("csv")
    .option("skipRows", 1)
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .load(caminho_csv)
)
display(df_csv)

In [0]:
caminho_csv = "dbfs:/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA.csv"

df_csv.write \
    .option("compression", "gzip") \
    .option("header", "true") \
    .option("sep", ";") \
    .mode("overwrite") \
    .csv("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA-zip")


In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/"))

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA.csv"))
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA-zip"))

## Lendo Arquivo reduzido gzip

In [0]:
df_novo = spark.read \
    .option("compression", "gzip") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ";") \
    .csv("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA-zip")

display(df_novo)


In [0]:
df_csv.printSchema()

In [0]:
# Converter o DataFrame do Spark para pandas
df_pandas = df_csv.toPandas()

# Exibir as primeiras linhas como tabela
display(df_pandas.head(20))

In [0]:
df = spark.read.json("/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA.json")
display(df)

In [0]:
df = df.withColumnRenamed("Aerodromo_de_Destino", "Destino") \
       .withColumnRenamed("Aerodromo_de_Origem", "Origem") \
       .withColumnRenamed("Classificacao_da_Ocorrência", "Classificacao") \
       .withColumnRenamed("Descricao_do_Tipo", "Tipo_de_Descricao") 
display(df)


In [0]:
df.write \
    .format("json") \
    .option("compression", "gzip") \
    .mode("overwrite") \
    .save("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/json_zip")

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/json_zip"))

In [0]:
arq = "dbfs:/Volumes/workspace/default/arquivos-aula/Anac/json_zip"
df = (
    spark.read
    .option("compression", "gzip")
    .json(arq)
)
display(df)

## Transformando DataFrame em Parquet

In [0]:
df = spark.read.json("/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA.json")

(
  df.write
  .format("parquet")
  .mode("overwrite")
  .save("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/parquet")
)

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/parquet/"))

In [0]:
dfpq = spark.read.parquet("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/parquet/")
display(dfpq)

In [0]:
display(dfpq.select("Classificacao_da_Ocorrência").distinct())

In [0]:
(
    dfpq.write
    .partitionBy("Classificacao_da_Ocorrência")
    .mode("overwrite")
    .parquet("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/parquet_particionado")
)

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/parquet_particionado/"))

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/parquet_particionado/Classificacao_da_Ocorrência=Acidente/"))

In [0]:
df_ocorr = spark.read.parquet("dbfs:/Volumes/workspace/default/arquivos-aula/Anac/parquet_particionado/Classificacao_da_Ocorrência=Acidente")
display(df_ocorr)